In [ ]:
import cvxpy as cp
import numpy as np
import pandas as pd
import json
import datetime
import matplotlib.pyplot as plt
import os
from matplotlib.patches import Patch
parent_dir = os.path.dirname(os.getcwd())
path = parent_dir 
save_path = parent_dir + '/'
from optimization import *
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

## Figure A.4

In [ ]:
period_string = '2024-06-03_to_2025-06-01'

num_clusters = 15
cluster_labels = np.array(pd.read_csv('Data/Cluster_Results_Adaptive/cluster_labels_'+str(num_clusters)+'.csv', header=None).iloc[:,0])
vinids = np.array(pd.read_csv('Data/vinids.csv').iloc[:,1])
color_palette = ['#332288',  '#117733', '#44AA99', '#88CCEE', '#DDCC77', '#CC6677', '#AA4499', '#882255']

In [ ]:
def collect_aggregate_data(results_path, week):
    # load results into matrices
    power_uncontrolled = []
    power_no_v2g = []
    power_v2g_home = []
    power_v2g_limited = []
    vin_not_found_idx = []

    for vin in range(1,750):
        try:
            power_uncontrolled.append( pd.read_csv(results_path + 'uncontrolled_power_driver_' +str(vin) + 'week_' +str(week) + '.csv'))
            power_no_v2g.append( pd.read_csv(results_path + 'power_managed_driver_' +str(vin) + 'week_' +str(week) + '.csv'))
            power_v2g_home.append( pd.read_csv(results_path + 'power_v2g_driver_' +str(vin) + 'week_' +str(week) + '.csv'))
            power_v2g_limited.append( pd.read_csv(results_path + 'power_v2g_limited_driver_' +str(vin) + 'week_' +str(week) + '.csv'))
        except:
            vin_not_found_idx.append(vin)
    
    power_mat_uncontrolled = np.array(power_uncontrolled)
    power_mat_no_v2g = np.array(power_no_v2g)
    power_mat_v2g_home = np.array(power_v2g_home)
    power_mat_v2g_limited = np.array(power_v2g_limited)

    return power_mat_uncontrolled, power_mat_no_v2g, power_mat_v2g_home, power_mat_v2g_limited

In [ ]:
fig, ax = plt.subplots(3,1, figsize=(11, 12))
labels = ['a.', 'b.', 'c.']
for week in range(3):
    results_path = 'Results/cir_1/batt_aging_0/power/batt_aging_0/elrp_1_'
    power_uncontrolled, power_no_v2g, power_v2g_home, power_v2g_limited = collect_aggregate_data(results_path, week)
    ax[week].plot(np.sum(power_uncontrolled, axis=0), color = color_palette[0], label='Baseline Charging', linewidth=1.5)
    ax[week].plot(np.sum(power_no_v2g, axis=0), color = color_palette[1], label='Managed Charging', linewidth=1.5)
    ax[week].plot(np.sum(power_v2g_home, axis=0), color = color_palette[3], label='V2G Home', linewidth=1.5)
    ax[week].plot(np.sum(power_v2g_limited, axis=0), color = color_palette[7], label='V2G Home Limited', linewidth=1.5)
    ax[week].set_xlabel('Time [day]', fontsize=13)
    #set x axis ticks to days
    ax[week].set_xticks(np.arange(0, 24*7*60+1, 24*60))
    ax[week].set_xticklabels(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday', 'Monday'])
    ax[week].annotate('Week '+str(week+1), xy=(0.02, 0.9), xycoords='axes fraction', fontsize=14)
    ax[week].annotate(labels[week], xy=(-0.09, 1.05), xycoords='axes fraction', fontsize=14)
    ax[week].set_ylabel('Power [MW]', fontsize=15)
    ax[week].set_yticklabels(np.round(ax[week].get_yticks()/1000,2), fontsize=13)
    ax[week].tick_params(axis='both', which='major', labelsize=13)
    #line at y=0
    ax[week].axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.legend(fontsize=13, ncols=2, loc='upper right')
plt.savefig(save_path + 'A4_load_profiles.pdf', bbox_inches='tight')